In [4]:
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike

d:\College\7th sem\campus-care\apps\ml-service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
Settings.llm = OpenAILike(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    api_base="https://api.groq.com/openai/v1",
    is_chat_model=True,
    context_window=131072,
)


In [7]:
# set the embed model
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

In [21]:
import fitz
from llama_index.core import Document

pdf_path = "./ml_service/data/Understanding-Mental-Health.pdf"

pdf = fitz.open(pdf_path)

documents = []

for page_number, page in enumerate(pdf):
    text = page.get_text()

    if text.strip():
        documents.append(
            Document(
                text=text,
                metadata={
                    "file_name": "Understanding-Mental-Health.pdf",
                    "page_number": page_number + 1,
                },
            )
        )

print("Documents:", len(documents))
print(documents[0].text[:2000])

Documents: 3
UNDERSTANDING MENTAL HEALTH
WHAT IS MENTAL HEALTH?
Our mental health directly influences how we think, feel and act: it also affects our physical health. Work, in fact, is actually one of 
the best things for protecting our mental health, but it can also adversely affect it. 
Good mental health and well-being is not an on-off 
experience. We can all have days, weeks or months 
where we feel resilient, strong and optimistic, 
regardless of events or situations. Often that can 
be mixed with or shift to a very different set of 
thoughts, feelings and behaviours; or not feeling 
resilient and optimistic in just one or two areas of 
our life.  For about twenty-five per cent of us, that 
may shift to having a significant impact on how 
we think, feel and act in many parts of our lives, 
including relationships, experiences at work, sense 
of connection to peer groups and our personal sense 
of worth, physical health and motivation. This could 
lead to us developing a mental hea

In [22]:
index = VectorStoreIndex.from_documents(
    documents, show_progress= True
)
index.storage_context.persist("./ml_service/storage")

Generating embeddings: 100%|██████████| 3/3 [00:00<00:00,  5.24it/s]


In [23]:
# --- Query ---
query_engine = index.as_query_engine()
response = query_engine.query(
    "I am feeling stressed"
)

print("Response:")
print(response)

print("\nSources:")
for source in response.source_nodes:
    print("\n---")
    print(source.text)

Response:
You are not alone, as many people experience stress. It's a normal part of life, and it can affect anyone. When we're stressed, it can impact our mental health and wellbeing. It's essential to recognize that mental health exists on a continuum, and it's possible to move back and forth along this range during our lifetime in response to different stressors and circumstances. If you're finding it difficult to cope, it may be helpful to seek support from a mental health professional, such as a psychologist or psychiatrist, who can provide you with guidance and treatment options, like therapy or medication, including antidepressants if necessary.

Sources:

---
UNDERSTANDING MENTAL HEALTH
WHAT IS MENTAL HEALTH?
Our mental health directly influences how we think, feel and act: it also affects our physical health. Work, in fact, is actually one of 
the best things for protecting our mental health, but it can also adversely affect it. 
Good mental health and well-being is not an on-